In [ ]:
import praw
import json
import csv
import re
import string
import datetime
import pandas as pd

In [ ]:
with open('client_secrets.json', 'r') as secrets_file:
    secrets = json.load(secrets_file)

reddit = praw.Reddit(
    client_id=secrets['client_id'],
    client_secret=secrets['client_secret'],
    refresh_token=secrets['refresh_token'],
    user_agent='MyRedditApp/1.0'
)

In [ ]:
def preprocess_text_simple(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-zżźćńółęąśŻŹĆĄŚĘŁÓŃ0-9\s]", "", text)
    text = re.sub(f"[{re.escape(string.punctuation)}]", '', text)
    return text

In [ ]:
csvfile = open('activity_today2.csv', mode='w', newline='')
writer = csv.writer(csvfile, delimiter=',')
writer.writerow(['author', 'date', 'type', 'subreddit', 'content', 'ratio', 'score', 'replies'])

In [ ]:
def get_data(subreddit, hashtags, limit=None):
    contributors = set()
    global writer
    for submission in subreddit.search(" OR ".join(hashtags), limit=limit):
        
        data = submission.created_utc
        data = datetime.datetime.fromtimestamp(data)
        if data < datetime.datetime(2023, 4, 1):
            continue
            
        if submission.author:
            contributors.add(submission.author.name)
            
        writer.writerow(
            [submission.author, data, 'submission', submission.subreddit, preprocess_text_simple(submission.title),
             submission.upvote_ratio, submission.score, submission.num_comments])
        
        submission.comments.replace_more(limit=0)
        for comment in submission.comments.list():
            if comment.author:
                contributors.add(comment.author.name)
                writer.writerow([comment.author, data, 'comment', comment.subreddit, preprocess_text_simple(comment.body), 1, comment.score,len(comment.replies)])
                
    return contributors

In [ ]:
subreddits = ['PolskaPolityka', 'Polska', 'libek', 'MapPorn']
limit = 100

In [ ]:
# read keywords
keywords = []
with open('keywords.txt', 'r') as keywords_file:
    for line in keywords_file:
        keywords.append(line.strip())

In [ ]:
print('Keywords: ', len(keywords))

In [ ]:
for sub_name in subreddits:
    print('Subreddit: ', sub_name)
    for i in range(0, len(keywords), 5):
        users = get_data(reddit.subreddit(sub_name), keywords[i:i+5])
        print('Users: ', len(users))

In [ ]:
csvfile.close()

In [ ]:
# read data 
df = pd.read_csv('activity_today2.csv')
df.head()

In [ ]:
df.shape

In [ ]:
# get distinct users
df['author'].nunique()

In [ ]:
# get data range
df['date'].min(), df['date'].max()

In [ ]:
# delete duplicates
df.drop_duplicates(inplace=True)
df.shape

In [ ]:
# save data
df.to_csv('activity_today_dist2.csv', index=False)

In [ ]:
# print unique subreddits
df['subreddit'].unique()